<a href="https://colab.research.google.com/github/SJSU-CMPE259/AI-Stock-Research-and-Earnings-Assistant/blob/p1%2Fedgar-xbrl/notebooks/p1_edgar_xbrl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinSight · Sprint 1: EDGAR and XBRL tools

**Owner:** P1 (Data) · **Sprint demo:** fetch a 10-Q and its XBRL figures by ticker

This notebook builds the two data tools FinSight's agents depend on:

| Tool | What it answers | Why we need it |
|---|---|---|
| **EDGAR** | *Which filings did this company file, and where are they?* | The source documents (10-K, 10-Q, 8-K) for RAG and citations |
| **XBRL** | *What exact number did the company report for revenue in FY2026 Q3?* | Reliable numbers without an LLM reading tables, for the ≥ 95% numeric accuracy target |

Both use free SEC APIs with no key. The SEC asks for two things:
1. A **User-Agent** that identifies you (name and email). Without it, you get HTTP 403.
2. **At most 10 requests per second.** The client below stays under that and caches responses.

> This is the exploration version. The same code lives in `src/finsight/tools/` in the repo, and that's the version the agents import.

## 1. Setup

In [ ]:
# Colab already has these; this is for running locally
#%pip install -q requests pandas matplotlib

In [ ]:
import os

# Put your real name/email here. The SEC rejects anonymous requests.
os.environ["SEC_USER_AGENT"] = "FinSight CMPE259 Team1 your_email@sjsu.edu"

assert "your_email" not in os.environ["SEC_USER_AGENT"], "Edit SEC_USER_AGENT above first"


## 2. SEC client (shared by both tools)

Every request goes through `get_json` / `get_text`, which:
- send the User-Agent header
- wait between requests (rate limit) and retry on 429 / 5xx errors
- cache responses in `data/cache/sec/`, so rerunning a cell doesn't hit the SEC again

In [ ]:
"""Shared HTTP client for SEC EDGAR.

The SEC requires two things from every caller:
  1. A User-Agent header that identifies you, e.g. "FinSight Team1 you@sjsu.edu".
     Requests without it get HTTP 403.
  2. No more than 10 requests per second. We stay well under that.

Responses are cached on disk so repeated runs (and teammates' notebooks)
don't hit the SEC again for the same data.
"""

import hashlib
import json
import os
import time
from pathlib import Path

import requests

USER_AGENT = os.getenv("SEC_USER_AGENT", "")
CACHE_DIR = Path(os.getenv("FINSIGHT_CACHE_DIR", "data/cache/sec"))
CACHE_TTL_SECONDS = 24 * 60 * 60  # refetch after a day so new filings show up
MIN_INTERVAL = 0.15  # seconds between requests (about 6-7 per second)

_last_request = 0.0
_session = requests.Session()


def _headers() -> dict:
    if not USER_AGENT:
        raise RuntimeError(
            "Set SEC_USER_AGENT in your .env, e.g. 'FinSight Team1 you@sjsu.edu'"
        )
    return {"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate"}


def _cache_path(url: str, suffix: str) -> Path:
    key = hashlib.sha256(url.encode()).hexdigest()[:24]
    return CACHE_DIR / f"{key}{suffix}"


def _fetch(url: str) -> requests.Response:
    global _last_request
    wait = MIN_INTERVAL - (time.time() - _last_request)
    if wait > 0:
        time.sleep(wait)
    for attempt in range(3):
        _last_request = time.time()
        resp = _session.get(url, headers=_headers(), timeout=30)
        if resp.status_code == 429 or resp.status_code >= 500:
            time.sleep(2 ** attempt)  # back off and retry
            continue
        resp.raise_for_status()
        return resp
    resp.raise_for_status()
    return resp


def _fresh(path: Path, ttl: int) -> bool:
    return path.exists() and (time.time() - path.stat().st_mtime) < ttl


def get_json(url: str, ttl: int = CACHE_TTL_SECONDS) -> dict:
    path = _cache_path(url, ".json")
    if _fresh(path, ttl):
        return json.loads(path.read_text())
    data = _fetch(url).json()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data))
    return data


def get_text(url: str, ttl: int = 365 * 24 * 60 * 60) -> str:
    """Filing documents never change once filed, so cache them for a long time."""
    path = _cache_path(url, ".html")
    if _fresh(path, ttl):
        return path.read_text(errors="replace")
    text = _fetch(url).text
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
    return text

## 3. EDGAR tool: find filings

Steps:
1. **Ticker → CIK.** The SEC identifies companies by a 10-digit CIK, not by ticker.
2. **CIK → filing list** from `data.sec.gov/submissions/CIK##########.json`.
3. **Filing → document URL** in the EDGAR archive.

The **accession number** (e.g. `0001045810-25-000230`) is each filing's unique ID. We store it with every number and every chunk of text so the Report agent can cite it.

In [ ]:
"""EDGAR tool: find a company's filings (10-K, 10-Q, 8-K) and fetch the documents.

EDGAR is the SEC's database of every filing a public company submits.
This module answers questions like "what is NVIDIA's latest 10-Q and where is it?"

Data sources (all free, no API key):
  - https://www.sec.gov/files/company_tickers.json          ticker -> CIK
  - https://data.sec.gov/submissions/CIK##########.json     list of filings
  - https://www.sec.gov/Archives/edgar/data/<cik>/<acc>/<doc>  the filing itself
"""

from dataclasses import asdict, dataclass


TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
SUBMISSIONS_URL = "https://data.sec.gov/submissions/CIK{cik}.json"
ARCHIVE_URL = "https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc_nodash}/{doc}"


@dataclass
class Filing:
    ticker: str
    cik: str              # 10-digit, zero-padded
    form: str             # "10-Q", "10-K", "8-K", ...
    accession: str        # e.g. "0001045810-25-000230" — the filing's unique ID
    filing_date: str      # when it was filed with the SEC
    period_of_report: str # the period end date the filing covers
    primary_document: str
    url: str

    def to_dict(self) -> dict:
        return asdict(self)


def get_cik(ticker: str) -> str:
    """Return the 10-digit CIK for a ticker, e.g. 'NVDA' -> '0001045810'."""
    data = get_json(TICKERS_URL, ttl=7 * 24 * 60 * 60)
    t = ticker.upper()
    for row in data.values():
        if row["ticker"].upper() == t:
            return str(row["cik_str"]).zfill(10)
    raise ValueError(f"Ticker not found in SEC list: {ticker}")


def get_filings(ticker: str, form: str = "10-Q", limit: int = 4) -> list[Filing]:
    """Most recent filings of one form type, newest first.

    Uses the 'recent' block of the submissions API (about the last 1,000
    filings), which covers several years for most companies.
    """
    cik = get_cik(ticker)
    recent = get_json(SUBMISSIONS_URL.format(cik=cik))["filings"]["recent"]
    out: list[Filing] = []
    for i, f in enumerate(recent["form"]):
        if f != form:
            continue
        acc = recent["accessionNumber"][i]
        doc = recent["primaryDocument"][i]
        out.append(
            Filing(
                ticker=ticker.upper(),
                cik=cik,
                form=f,
                accession=acc,
                filing_date=recent["filingDate"][i],
                period_of_report=recent["reportDate"][i],
                primary_document=doc,
                url=ARCHIVE_URL.format(
                    cik_int=int(cik), acc_nodash=acc.replace("-", ""), doc=doc
                ),
            )
        )
        if len(out) >= limit:
            break
    return out


def get_latest_filing(ticker: str, form: str = "10-Q") -> Filing:
    filings = get_filings(ticker, form, limit=1)
    if not filings:
        raise ValueError(f"No {form} found for {ticker}")
    return filings[0]


def get_filing_html(filing: Filing) -> str:
    """Raw HTML of the filing's main document (cleaning it is a Sprint 2 job)."""
    return get_text(filing.url)

### Try it

In [ ]:
import pandas as pd

TICKER = "NVDA"
print("CIK:", get_cik(TICKER))

filings = get_filings(TICKER, "10-Q", limit=4)
pd.DataFrame([f.to_dict() for f in filings])[["form", "period_of_report", "filing_date", "accession", "url"]]

In [ ]:
# Latest 10-K and recent 8-Ks
latest_10k = get_latest_filing(TICKER, "10-K")
print("Latest 10-K:", latest_10k.period_of_report, latest_10k.url)

pd.DataFrame([f.to_dict() for f in get_filings(TICKER, "8-K", limit=5)])[["filing_date", "accession", "url"]]

In [ ]:
# Peek at the raw filing HTML (cleaning and chunking it is Sprint 2)
html = get_filing_html(filings[0])
print(f"{len(html):,} characters of HTML")
print(html[:500])

## 4. XBRL tool: get exact reported numbers

`companyfacts` returns every tagged number the company has ever reported. Each one carries:
`start`, `end` (the period), `val`, `accn` (the filing), `form`, `filed`, and **`fy` / `fp`**.

### ⚠️ The fiscal-quarter trap (the prototype's bug)
- Many companies' fiscal years don't match the calendar. NVIDIA's FY2026 Q3 ends in **October 2025**. The `frame` field (`CY2025Q3`) is calendar-based, so we never use it for labels.
- `fy` / `fp` describe the **filing**, not the number. A 10-Q also repeats last year's quarter for comparison, and that old number carries the **new** filing's `fy`/`fp`.
- **Rule:** trust `fy`/`fp` only on the number whose period end matches the filing's own (latest) period end.
- A 10-Q reports both 3-month and year-to-date values. For quarters, keep only the ~90-day ones.
- Companies don't file a Q4 10-Q. Q4 = FY − (Q1 + Q2 + Q3), computed in code.

In [ ]:
"""XBRL tool: get exact reported numbers (revenue, net income, ...) by fiscal period.

XBRL is the machine-readable version of the numbers in a filing. Every value
is tagged with a concept name (e.g. us-gaap:Revenues), a period, a unit and
the filing it came from. Using these instead of having the LLM read tables
is how FinSight hits its ">= 95% numeric accuracy" target.

Source: https://data.sec.gov/api/xbrl/companyfacts/CIK##########.json

Fiscal-quarter labels (the prototype's calendar-quarter bug):
  Each fact carries `fy` and `fp` (e.g. 2026, "Q3"), but those describe the
  FILING, not the fact. A 10-Q also repeats last year's numbers for
  comparison, and those carry the same fy/fp. So we only trust fy/fp for the
  fact whose period END matches the filing's own (latest) period end.
  We never use the `frame` field (e.g. CY2025Q3) for labels — that is calendar-based.
"""

from dataclasses import asdict, dataclass
from datetime import date


FACTS_URL = "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"

# Companies tag the same idea with different concepts. Try them in order.
METRICS: dict[str, list[str]] = {
    "revenue": [
        "Revenues",
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "SalesRevenueNet",
    ],
    "net_income": ["NetIncomeLoss"],
    "operating_income": ["OperatingIncomeLoss"],
    "gross_profit": ["GrossProfit"],
    "eps_diluted": ["EarningsPerShareDiluted"],
    "operating_cash_flow": ["NetCashProvidedByUsedInOperatingActivities"],
    "rd_expense": ["ResearchAndDevelopmentExpense"],
    "total_assets": ["Assets"],          # balance-sheet item: point in time
    "cash": ["CashAndCashEquivalentsAtCarryingValue"],
}


@dataclass
class FactValue:
    ticker: str
    metric: str
    concept: str
    fiscal_year: int
    fiscal_period: str   # "Q1", "Q2", "Q3", "FY"
    start: str | None    # None for point-in-time items like Assets
    end: str
    value: float
    unit: str
    form: str
    accession: str       # links the number back to the exact filing (for citations)
    filed: str

    @property
    def label(self) -> str:
        return f"FY{self.fiscal_year} {self.fiscal_period}"

    def to_dict(self) -> dict:
        d = asdict(self)
        d["label"] = self.label
        return d


def _days(start: str, end: str) -> int:
    return (date.fromisoformat(end) - date.fromisoformat(start)).days


def get_company_facts(ticker: str) -> dict:
    return get_json(FACTS_URL.format(cik=get_cik(ticker)))


def extract_metric(facts: dict, ticker: str, metric: str,
                   period: str = "quarterly") -> list[FactValue]:
    """Pure function (no network) so it can be unit-tested on a fixture.

    period="quarterly": 3-month values from 10-Qs (Q1-Q3).
    period="annual":    12-month values from 10-Ks (FY).
    Note: companies don't file a Q4 10-Q. Q4 = FY - (Q1+Q2+Q3), computed in code later.
    """
    gaap = facts.get("facts", {}).get("us-gaap", {})
    concepts = METRICS.get(metric, [metric])  # also accept a raw concept name

    for concept in concepts:
        if concept not in gaap:
            continue
        units = gaap[concept]["units"]
        unit = "USD" if "USD" in units else next(iter(units))
        rows = units[unit]

        # Latest period end in each filing = that filing's own period.
        latest_end: dict[str, str] = {}
        for r in rows:
            acc = r["accn"]
            if acc not in latest_end or r["end"] > latest_end[acc]:
                latest_end[acc] = r["end"]

        results: dict[tuple, FactValue] = {}
        for r in rows:
            form = r.get("form", "")
            if r["end"] != latest_end[r["accn"]]:
                continue  # comparative prior-period number; fy/fp would be wrong
            start = r.get("start")
            if start:  # duration item (income statement, cash flow)
                d = _days(start, r["end"])
                if period == "quarterly" and not (form.startswith("10-Q") and 80 <= d <= 100):
                    continue
                if period == "annual" and not (form.startswith("10-K") and 350 <= d <= 380):
                    continue
            else:  # point-in-time item (balance sheet)
                want = "10-Q" if period == "quarterly" else "10-K"
                if not form.startswith(want):
                    continue

            fv = FactValue(
                ticker=ticker.upper(), metric=metric, concept=concept,
                fiscal_year=int(r["fy"]), fiscal_period=r["fp"],
                start=start, end=r["end"], value=r["val"], unit=unit,
                form=form, accession=r["accn"], filed=r["filed"],
            )
            key = (fv.start, fv.end)
            # If a later amendment (10-Q/A) restates it, keep the newest filing.
            if key not in results or fv.filed > results[key].filed:
                results[key] = fv

        if results:
            return sorted(results.values(), key=lambda x: x.end, reverse=True)
    return []


def get_metric(ticker: str, metric: str, period: str = "quarterly",
               limit: int = 8) -> list[FactValue]:
    """e.g. get_metric("NVDA", "revenue") -> last 8 fiscal quarters, newest first."""
    return extract_metric(get_company_facts(ticker), ticker, metric, period)[:limit]

### Check the labeling logic offline
This sample uses **made-up values** in NVIDIA's fiscal calendar and needs no internet. It's the same fixture the repo's tests use.

In [ ]:
SAMPLE_FACTS = {
 "cik": 1045810,
 "entityName": "SAMPLE (NVIDIA-like fiscal calendar, made-up values)",
 "facts": {
  "us-gaap": {
   "Revenues": {
    "units": {
     "USD": [
      {
       "start": "2024-07-29",
       "end": "2024-10-27",
       "val": 35082000000,
       "accn": "0001045810-24-000316",
       "fy": 2025,
       "fp": "Q3",
       "form": "10-Q",
       "filed": "2024-11-20",
       "frame": "CY2024Q3"
      },
      {
       "start": "2024-01-29",
       "end": "2024-10-27",
       "val": 91166000000,
       "accn": "0001045810-24-000316",
       "fy": 2025,
       "fp": "Q3",
       "form": "10-Q",
       "filed": "2024-11-20"
      },
      {
       "start": "2024-01-29",
       "end": "2025-01-26",
       "val": 130497000000,
       "accn": "0001045810-25-000023",
       "fy": 2025,
       "fp": "FY",
       "form": "10-K",
       "filed": "2025-02-26",
       "frame": "CY2024"
      },
      {
       "start": "2023-01-30",
       "end": "2024-01-28",
       "val": 60922000000,
       "accn": "0001045810-25-000023",
       "fy": 2025,
       "fp": "FY",
       "form": "10-K",
       "filed": "2025-02-26"
      },
      {
       "start": "2025-07-28",
       "end": "2025-10-26",
       "val": 57006000000,
       "accn": "0001045810-25-000230",
       "fy": 2026,
       "fp": "Q3",
       "form": "10-Q",
       "filed": "2025-11-19",
       "frame": "CY2025Q3"
      },
      {
       "start": "2024-07-29",
       "end": "2024-10-27",
       "val": 35082000000,
       "accn": "0001045810-25-000230",
       "fy": 2026,
       "fp": "Q3",
       "form": "10-Q",
       "filed": "2025-11-19"
      },
      {
       "start": "2025-01-27",
       "end": "2025-10-26",
       "val": 147811000000,
       "accn": "0001045810-25-000230",
       "fy": 2026,
       "fp": "Q3",
       "form": "10-Q",
       "filed": "2025-11-19"
      }
     ]
    }
   },
   "Assets": {
    "units": {
     "USD": [
      {
       "end": "2025-10-26",
       "val": 161148000000,
       "accn": "0001045810-25-000230",
       "fy": 2026,
       "fp": "Q3",
       "form": "10-Q",
       "filed": "2025-11-19"
      },
      {
       "end": "2025-01-26",
       "val": 111601000000,
       "accn": "0001045810-25-000230",
       "fy": 2026,
       "fp": "Q3",
       "form": "10-Q",
       "filed": "2025-11-19"
      }
     ]
    }
   }
  }
 }
}

rows = extract_metric(SAMPLE_FACTS, "NVDA", "revenue", "quarterly")
for r in rows:
    print(r.label, r.end, r.value, r.accession)

assert [r.label for r in rows] == ["FY2026 Q3", "FY2025 Q3"]
assert rows[1].accession == "0001045810-24-000316"  # labeled from its own filing
print("\n✓ fiscal labels correct; prior-year comparative not mislabeled")

### Try it on live data

In [ ]:
def metric_table(ticker, metric, period="quarterly", limit=8):
    rows = get_metric(ticker, metric, period, limit)
    df = pd.DataFrame([r.to_dict() for r in rows])
    if df.empty:
        print(f"No data for {ticker} {metric}; it may use a different concept name (see METRICS)")
        return df
    df["value_B"] = (df["value"] / 1e9).round(2)
    return df[["label", "fiscal_year", "fiscal_period", "end", "value_B", "concept", "form", "accession"]]

metric_table(TICKER, "revenue")

In [ ]:
metric_table(TICKER, "net_income")

In [ ]:
metric_table(TICKER, "revenue", "annual", limit=5)

### Compute changes in code (never by the LLM)
Changes are computed from the XBRL values, as US-05 requires. We use **year-over-year** (the same fiscal quarter one year earlier), because Q4 has no 10-Q, so the previous row isn't always the previous quarter.

In [ ]:
rev = metric_table(TICKER, "revenue", limit=12).iloc[::-1].reset_index(drop=True)  # oldest first

prev = rev[["fiscal_year", "fiscal_period", "value_B"]].copy()
prev["fiscal_year"] += 1  # last year's value lines up with this year's same quarter
rev = rev.merge(prev, on=["fiscal_year", "fiscal_period"], how="left", suffixes=("", "_prev_year"))
rev["YoY_%"] = ((rev["value_B"] / rev["value_B_prev_year"] - 1) * 100).round(1)
rev[["label", "end", "value_B", "value_B_prev_year", "YoY_%"]]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(rev["label"], rev["value_B"], color="#2b6cb0")
ax.set_ylabel("Revenue ($B)")
ax.set_title(f"{TICKER} quarterly revenue (fiscal quarters, from XBRL)")
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Sprint 1 demo: run all 10 companies
Replace the list with your team's 10 tickers. If revenue comes back empty for a company, find its concept name in `get_company_facts(ticker)["facts"]["us-gaap"].keys()` and add it to `METRICS["revenue"]`.

In [ ]:
COMPANIES = ["NVDA", "AAPL", "MSFT", "AMZN", "GOOGL", "META", "TSLA", "JPM", "AMD", "INTC"]  # ← your 10

summary = []
for t in COMPANIES:
    try:
        f = get_latest_filing(t, "10-Q")
        rev_rows = get_metric(t, "revenue", limit=1)
        ni_rows = get_metric(t, "net_income", limit=1)
        summary.append({
            "ticker": t,
            "latest_10Q_period": f.period_of_report,
            "filed": f.filing_date,
            "label": rev_rows[0].label if rev_rows else None,
            "revenue_B": round(rev_rows[0].value / 1e9, 2) if rev_rows else None,
            "net_income_B": round(ni_rows[0].value / 1e9, 2) if ni_rows else None,
            "revenue_concept": rev_rows[0].concept if rev_rows else "MISSING",
        })
    except Exception as e:
        summary.append({"ticker": t, "error": str(e)})

pd.DataFrame(summary)

**Check before the demo:** for each company, compare one number against the press release or the 10-Q itself. Make sure the label (e.g. FY2026 Q3) matches what the company calls that quarter.

## Next steps
- Move any changes back into `src/finsight/tools/` in the repo and open a PR.
- Q4 values: FY − (Q1+Q2+Q3).
- Save results into shared memory (CAVM) as named variables, e.g. `NVDA_xbrl_revenue` (with P4).
- Sprint 2: clean the filing HTML, split it into sections (Item 1A, Item 7…), then chunk and embed it.